
# 01B — Penerapan Keputusan Review Manual dan Penggabungan Awal

Notebook ini menjadi **bukti proses di antara preprocessing per anggota dan deduplikasi lintas anggota**.

## Posisi dalam pipeline

```text
Preprocessing Dwi, Indra, dan Rajif
        ↓
Kandidat KEEP dan REVIEW
        ↓
Review manual kandidat REVIEW
        ↓
Keputusan disimpan pada Master_Keputusan_Audit_Entitas.csv
        ↓
Notebook ini menerapkan keputusan KEEP/DROP
        ↓
Gabungan 3.464 baris sebelum deduplikasi lintas anggota
        ↓
02_Merger_dan_Audit_Lintas_Anggota_Final.ipynb
```

## Prinsip metodologis

- Kode **tidak menentukan status UMKM secara hukum**.
- Kode preprocessing hanya menandai kandidat yang memerlukan review.
- Keputusan terhadap kandidat dibuat melalui review manual berdasarkan kriteria operasional penelitian.
- Keputusan manual disimpan dalam `Master_Keputusan_Audit_Entitas.csv`.
- Notebook ini menerapkan keputusan secara terprogram agar konsisten, dapat direproduksi, dan dapat diaudit.
- Data `KEEP` dari ketiga kelompok digabungkan, tetapi **belum dideduplikasi lintas anggota** pada notebook ini.


In [ ]:

# ============================================================
# 1. IMPORT LIBRARY
# ============================================================
import os
import zipfile
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

try:
    from google.colab import files
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

print("Library berhasil dimuat.")


In [ ]:

# ============================================================
# 2. KONFIGURASI FILE
# ============================================================
INPUT_FILES = {
    "Dwi": "01_Hasil_Preprocessing_Dwi_Final.csv",
    "Indra": "01_Hasil_Preprocessing_Indra_Final.csv",
    "Rajif": "01_Hasil_Preprocessing_Rajif_Final.csv",
}

MASTER_DECISION_FILE = "Master_Keputusan_Audit_Entitas.csv"
INPUT_SHEET = "Data_Siap_Merger"

OUTPUT_EVIDENCE_CSV = "01B_Bukti_Review_Entitas_dan_Penggabungan_Awal.csv"
OUTPUT_COMBINED_CSV = "02_Data_Reviewed_Gabungan_Sebelum_Dedup.csv"
OUTPUT_SUMMARY_CSV = "02_Ringkasan_Review.csv"
OUTPUT_ZIP = "01B_Paket_Bukti_Review_dan_Penggabungan_Awal.zip"

MODEL_COLUMNS = [
    "title",
    "totalScore",
    "reviewsCount",
    "street",
    "city",
    "categoryName",
    "text",
]

# Angka checkpoint berdasarkan snapshot preprocessing final saat ini.
# Jika input berubah, notebook memberi peringatan agar angka laporan tidak
# tercampur dengan versi dataset lain.
EXPECTED_COUNTS = {
    "Dwi": {
        "siap_merger_awal": 1408,
        "review": 42,
        "review_keep": 29,
        "review_drop": 13,
        "keep_final": 1395,
    },
    "Indra": {
        "siap_merger_awal": 968,
        "review": 22,
        "review_keep": 11,
        "review_drop": 11,
        "keep_final": 957,
    },
    "Rajif": {
        "siap_merger_awal": 1199,
        "review": 99,
        "review_keep": 12,
        "review_drop": 87,
        "keep_final": 1112,
    },
}

STRICT_EXPECTED_COUNTS = True

print("Konfigurasi selesai.")



## Kriteria operasional review

Kandidat `REVIEW` diperiksa dengan mempertimbangkan:

1. Apakah listing merepresentasikan satu usaha individual atau entitas agregat.
2. Apakah nama mengarah pada jaringan nasional/internasional, korporasi, institusi, dealer resmi, service center resmi, atau pusat kegiatan yang berisi banyak usaha.
3. Apakah kata seperti *pasar*, *mall*, *hotel*, *sekolah*, atau *square* hanya merupakan bagian dari nama produk/alamat tenant, atau benar-benar menunjukkan entitas agregat.
4. Apakah terdapat bukti bahwa usaha merupakan tenant/usaha independen sehingga layak dipertahankan.
5. Apakah entitas sesuai dengan definisi operasional usaha lokal yang digunakan penelitian.

Keputusan akhir hanya menggunakan:

- `KEEP`: dipertahankan dalam ruang lingkup penelitian.
- `DROP`: dikeluarkan berdasarkan kriteria eksklusi operasional.

Keputusan dan alasannya tersimpan pada `Master_Keputusan_Audit_Entitas.csv`.


In [ ]:

# ============================================================
# 3. TABEL KRITERIA OPERASIONAL UNTUK DOKUMENTASI
# ============================================================
kriteria_operasional = pd.DataFrame([
    {
        "kode": "KEEP_INDIVIDUAL",
        "keputusan": "KEEP",
        "kriteria": "Satu usaha individual atau tenant yang masih merepresentasikan satu unit usaha.",
        "contoh_penjelasan": "Kata pasar/mall/sekolah hanya berasal dari alamat, produk, atau nama toko."
    },
    {
        "kode": "DROP_JARINGAN_BESAR",
        "keputusan": "DROP",
        "kriteria": "Jaringan nasional/internasional, korporasi besar, atau waralaba yang dikecualikan.",
        "contoh_penjelasan": "Nama dan informasi publik menunjukkan jaringan dengan banyak cabang."
    },
    {
        "kode": "DROP_INSTITUSI",
        "keputusan": "DROP",
        "kriteria": "Institusi publik/korporat yang tidak sesuai dengan objek usaha lokal penelitian.",
        "contoh_penjelasan": "Bank, rumah sakit, sekolah, universitas, kantor perusahaan, atau fasilitas publik."
    },
    {
        "kode": "DROP_ENTITAS_AGREGAT",
        "keputusan": "DROP",
        "kriteria": "Listing merepresentasikan kawasan atau tempat yang berisi banyak unit usaha.",
        "contoh_penjelasan": "Pasar, mall, plaza, trade center, food court, atau kawasan kuliner."
    },
    {
        "kode": "DROP_UNIT_KORPORAT",
        "keputusan": "DROP",
        "kriteria": "Dealer resmi, service center resmi, hotel, klinik, atau unit bisnis yang dikeluarkan berdasarkan batasan operasional penelitian.",
        "contoh_penjelasan": "Dikeluarkan untuk menjaga konsistensi ruang lingkup objek penelitian."
    },
    {
        "kode": "REVIEW_AMBIGU",
        "keputusan": "REVIEW MANUAL",
        "kriteria": "Nama atau kategori belum cukup untuk membuat keputusan otomatis.",
        "contoh_penjelasan": "Diperiksa menggunakan nama, kategori, alamat, situs, dan catatan verifikasi."
    },
])

display(kriteria_operasional)


In [ ]:

# ============================================================
# 4. UPLOAD INPUT JIKA BELUM TERSEDIA
# ============================================================
required_files = list(INPUT_FILES.values()) + [MASTER_DECISION_FILE]
missing_files = [name for name in required_files if not os.path.exists(name)]

if missing_files:
    if not RUNNING_IN_COLAB:
        raise FileNotFoundError(
            "File berikut belum tersedia: " + ", ".join(missing_files)
        )

    print("Silakan upload seluruh file berikut:")
    for name in missing_files:
        print("-", name)

    uploaded = files.upload()

    remaining_missing = [
        name for name in required_files if not os.path.exists(name)
    ]

    if remaining_missing:
        raise FileNotFoundError(
            "Masih ada file yang belum ditemukan: "
            + ", ".join(remaining_missing)
        )

print("Seluruh file input ditemukan.")


In [ ]:

# ============================================================
# 5. FUNGSI PEMBACAAN MASTER KEPUTUSAN
# ============================================================
def read_csv_flexible(path):
    attempts = [
        {"sep": ";", "encoding": "utf-8-sig"},
        {"sep": ",", "encoding": "utf-8-sig"},
        {"sep": ";", "encoding": "utf-8"},
        {"sep": ",", "encoding": "utf-8"},
    ]

    errors = []

    for config in attempts:
        try:
            df_temp = pd.read_csv(path, **config)

            if df_temp.shape[1] > 1:
                return df_temp

        except Exception as exc:
            errors.append(str(exc))

    raise ValueError(
        f"Gagal membaca {path}. Percobaan parser: {errors}"
    )


master = read_csv_flexible(MASTER_DECISION_FILE)
master.columns = [str(col).strip() for col in master.columns]

required_master_columns = [
    "anggota",
    "entity_key",
    "keputusan_final",
    "alasan_final",
]

missing_master_columns = [
    col for col in required_master_columns if col not in master.columns
]

if missing_master_columns:
    raise ValueError(
        "Kolom master keputusan tidak lengkap: "
        + ", ".join(missing_master_columns)
    )

master["anggota"] = master["anggota"].astype(str).str.strip()
master["entity_key"] = master["entity_key"].astype(str).str.strip()
master["keputusan_final"] = (
    master["keputusan_final"]
    .astype(str)
    .str.strip()
    .str.upper()
)

allowed_decisions = {"KEEP", "DROP"}
invalid_master_decisions = master[
    ~master["keputusan_final"].isin(allowed_decisions)
].copy()

if len(invalid_master_decisions) > 0:
    raise ValueError(
        "Master memiliki keputusan selain KEEP/DROP."
    )

duplicate_master = master[
    master.duplicated(["anggota", "entity_key"], keep=False)
].copy()

if len(duplicate_master) > 0:
    raise ValueError(
        "Master memiliki pasangan anggota + entity_key yang duplikat."
    )

print("Jumlah keputusan manual pada master:", len(master))
display(
    master.groupby(
        ["anggota", "keputusan_final"]
    ).size().unstack(fill_value=0)
)


In [ ]:

# ============================================================
# 6. MEMBACA DATA SIAP MERGER DARI SETIAP ANGGOTA
# ============================================================
data_by_member = {}

required_data_columns = [
    "entity_key",
    "title",
    "totalScore",
    "reviewsCount",
    "street",
    "city",
    "categoryName",
    "text",
    "entity_decision",
    "entity_reason",
]

for anggota, path in INPUT_FILES.items():
    df_member = pd.read_excel(path, sheet_name=INPUT_SHEET)
    df_member.columns = [str(col).strip() for col in df_member.columns]

    missing_columns = [
        col for col in required_data_columns
        if col not in df_member.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{anggota}: kolom wajib tidak tersedia: {missing_columns}"
        )

    df_member.insert(0, "anggota", anggota)

    df_member["entity_key"] = (
        df_member["entity_key"]
        .astype(str)
        .str.strip()
    )

    df_member["entity_decision"] = (
        df_member["entity_decision"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    unexpected_decisions = df_member[
        ~df_member["entity_decision"].isin(["KEEP", "REVIEW"])
    ]

    if len(unexpected_decisions) > 0:
        raise ValueError(
            f"{anggota}: Data_Siap_Merger mengandung status selain KEEP/REVIEW."
        )

    data_by_member[anggota] = df_member

    print(
        anggota,
        "| total:", len(df_member),
        "| KEEP otomatis:", (df_member["entity_decision"] == "KEEP").sum(),
        "| REVIEW:", (df_member["entity_decision"] == "REVIEW").sum()
    )


In [ ]:

# ============================================================
# 7. VALIDASI CAKUPAN MASTER TERHADAP SELURUH KANDIDAT REVIEW
# ============================================================
coverage_records = []
unmatched_review_records = []
unused_master_records = []

for anggota, df_member in data_by_member.items():
    review_rows = df_member[
        df_member["entity_decision"] == "REVIEW"
    ][["anggota", "entity_key", "title", "street", "categoryName"]].copy()

    master_member = master[
        master["anggota"] == anggota
    ].copy()

    review_check = review_rows.merge(
        master_member[
            [
                "anggota",
                "entity_key",
                "keputusan_final",
                "alasan_final",
            ]
        ],
        on=["anggota", "entity_key"],
        how="left",
        validate="one_to_one"
    )

    unmatched = review_check[
        review_check["keputusan_final"].isna()
    ].copy()

    if len(unmatched) > 0:
        unmatched_review_records.append(unmatched)

    review_keys = set(review_rows["entity_key"])
    unused_master = master_member[
        ~master_member["entity_key"].isin(review_keys)
    ].copy()

    if len(unused_master) > 0:
        unused_master_records.append(unused_master)

    coverage_records.append({
        "anggota": anggota,
        "kandidat_review_dalam_data": len(review_rows),
        "keputusan_master_yang_cocok": review_check["keputusan_final"].notna().sum(),
        "review_tanpa_keputusan": review_check["keputusan_final"].isna().sum(),
        "keputusan_master_tidak_dipakai": len(unused_master),
    })

coverage_summary = pd.DataFrame(coverage_records)

if unmatched_review_records:
    unmatched_review = pd.concat(
        unmatched_review_records,
        ignore_index=True
    )
else:
    unmatched_review = pd.DataFrame()

if unused_master_records:
    unused_master = pd.concat(
        unused_master_records,
        ignore_index=True
    )
else:
    unused_master = pd.DataFrame()

display(coverage_summary)

if len(unmatched_review) > 0:
    display(unmatched_review)
    raise ValueError(
        "Ada kandidat REVIEW yang belum memiliki keputusan pada master."
    )

print(
    "Seluruh kandidat REVIEW pada Data_Siap_Merger "
    "sudah memiliki keputusan manual."
)
print(
    "Catatan: keputusan master yang tidak dipakai dapat berasal dari "
    "kandidat yang telah gugur pada tahap integrasi teks atau validasi lain."
)


In [ ]:

# ============================================================
# 8. MENERAPKAN KEPUTUSAN MANUAL
# ============================================================
def apply_manual_decisions(df_member, anggota, master_decisions):
    result = df_member.copy()

    manual_lookup = (
        master_decisions[
            master_decisions["anggota"] == anggota
        ]
        .set_index("entity_key")
    )

    result["final_decision"] = pd.NA
    result["final_reason"] = pd.NA
    result["decision_method"] = pd.NA
    result["decision_source"] = pd.NA

    auto_keep_mask = result["entity_decision"] == "KEEP"
    review_mask = result["entity_decision"] == "REVIEW"

    # Data yang tidak terkena flag otomatis tetap dipertahankan.
    result.loc[auto_keep_mask, "final_decision"] = "KEEP"
    result.loc[auto_keep_mask, "final_reason"] = (
        result.loc[auto_keep_mask, "entity_reason"]
        .fillna("Tidak terkena aturan otomatis.")
    )
    result.loc[auto_keep_mask, "decision_method"] = (
        "Keputusan otomatis preprocessing"
    )
    result.loc[auto_keep_mask, "decision_source"] = (
        "Notebook preprocessing final"
    )

    # Kandidat REVIEW menggunakan keputusan yang tercatat pada master.
    review_keys = result.loc[review_mask, "entity_key"]

    result.loc[review_mask, "final_decision"] = (
        review_keys.map(manual_lookup["keputusan_final"])
    )

    result.loc[review_mask, "final_reason"] = (
        review_keys.map(manual_lookup["alasan_final"])
    )

    result.loc[review_mask, "decision_method"] = (
        "Review manual berdasarkan kriteria operasional"
    )

    result.loc[review_mask, "decision_source"] = (
        MASTER_DECISION_FILE
    )

    if result["final_decision"].isna().any():
        raise ValueError(
            f"{anggota}: masih ada final_decision kosong."
        )

    if result["final_reason"].isna().any():
        raise ValueError(
            f"{anggota}: masih ada final_reason kosong."
        )

    invalid = result[
        ~result["final_decision"].isin(["KEEP", "DROP"])
    ]

    if len(invalid) > 0:
        raise ValueError(
            f"{anggota}: ditemukan final_decision tidak valid."
        )

    return result


reviewed_all_by_member = {}
kept_by_member = {}
dropped_by_member = {}

for anggota, df_member in data_by_member.items():
    reviewed = apply_manual_decisions(
        df_member,
        anggota,
        master
    )

    kept = reviewed[
        reviewed["final_decision"] == "KEEP"
    ].copy()

    dropped = reviewed[
        reviewed["final_decision"] == "DROP"
    ].copy()

    reviewed_all_by_member[anggota] = reviewed
    kept_by_member[anggota] = kept
    dropped_by_member[anggota] = dropped

    print(
        anggota,
        "| input:", len(reviewed),
        "| final KEEP:", len(kept),
        "| final DROP:", len(dropped)
    )


In [ ]:

# ============================================================
# 9. REKONSILIASI REVIEW PER ANGGOTA
# ============================================================
summary_records = []

for anggota in INPUT_FILES:
    df_input = data_by_member[anggota]
    df_reviewed = reviewed_all_by_member[anggota]
    df_keep = kept_by_member[anggota]
    df_drop = dropped_by_member[anggota]

    review_only = df_reviewed[
        df_reviewed["entity_decision"] == "REVIEW"
    ]

    summary_records.append({
        "anggota": anggota,
        "siap_merger_awal": len(df_input),
        "keep_otomatis_awal": (
            df_input["entity_decision"] == "KEEP"
        ).sum(),
        "kandidat_review": (
            df_input["entity_decision"] == "REVIEW"
        ).sum(),
        "review_diputuskan_keep": (
            (review_only["final_decision"] == "KEEP")
        ).sum(),
        "review_diputuskan_drop": (
            (review_only["final_decision"] == "DROP")
        ).sum(),
        "keep_final": len(df_keep),
        "drop_final": len(df_drop),
    })

ringkasan_review_lengkap = pd.DataFrame(summary_records)

ringkasan_review = ringkasan_review_lengkap[
    [
        "anggota",
        "siap_merger_awal",
        "keep_final",
        "drop_final",
    ]
].rename(
    columns={"drop_final": "drop_dari_review"}
)

display(ringkasan_review_lengkap)

print("\nTotal sebelum review :", ringkasan_review_lengkap["siap_merger_awal"].sum())
print("Total final KEEP     :", ringkasan_review_lengkap["keep_final"].sum())
print("Total final DROP     :", ringkasan_review_lengkap["drop_final"].sum())


In [ ]:

# ============================================================
# 10. VALIDASI CHECKPOINT ANGKA
# ============================================================
checkpoint_records = []
checkpoint_errors = []

for _, row in ringkasan_review_lengkap.iterrows():
    anggota = row["anggota"]
    expected = EXPECTED_COUNTS[anggota]

    actual_values = {
        "siap_merger_awal": int(row["siap_merger_awal"]),
        "review": int(row["kandidat_review"]),
        "review_keep": int(row["review_diputuskan_keep"]),
        "review_drop": int(row["review_diputuskan_drop"]),
        "keep_final": int(row["keep_final"]),
    }

    for indikator, actual in actual_values.items():
        target = expected[indikator]
        status = "SESUAI" if actual == target else "BERBEDA"

        checkpoint_records.append({
            "anggota": anggota,
            "indikator": indikator,
            "nilai_diharapkan": target,
            "nilai_aktual": actual,
            "status": status,
        })

        if actual != target:
            checkpoint_errors.append(
                f"{anggota} - {indikator}: "
                f"target {target}, aktual {actual}"
            )

checkpoint_validation = pd.DataFrame(checkpoint_records)
display(checkpoint_validation)

if checkpoint_errors:
    print("PERINGATAN CHECKPOINT:")
    for error in checkpoint_errors:
        print("-", error)

    if STRICT_EXPECTED_COUNTS:
        raise AssertionError(
            "Angka checkpoint berbeda dari snapshot final. "
            "Pastikan file input menggunakan versi yang benar."
        )
else:
    print("Semua angka checkpoint sesuai.")


In [ ]:

# ============================================================
# 11. MENGGABUNGKAN SELURUH DATA FINAL KEEP
# ============================================================
# Urutan anggota dipertahankan agar hasil dapat direproduksi.
member_order = ["Dwi", "Indra", "Rajif"]

df_gabungan = pd.concat(
    [kept_by_member[anggota] for anggota in member_order],
    ignore_index=True
)

# Kolom anggota dipertahankan untuk menunjukkan sumber pengumpulan data.
if "anggota" not in df_gabungan.columns:
    raise ValueError(
        "Kolom anggota tidak tersedia pada data gabungan."
    )

print("Jumlah gabungan sebelum deduplikasi lintas anggota:", len(df_gabungan))
print("\nDistribusi anggota sumber:")
display(
    df_gabungan["anggota"]
    .value_counts()
    .reindex(member_order)
    .rename("jumlah")
    .to_frame()
)

print("\nEntity key unik sementara:", df_gabungan["entity_key"].nunique())
print(
    "Baris duplikat lintas anggota yang masih mungkin ada:",
    df_gabungan.duplicated("entity_key").sum()
)

assert len(df_gabungan) == 3464, (
    "Jumlah gabungan bukan 3.464. Periksa versi input atau keputusan master."
)



## Makna file gabungan

File `02_Data_Reviewed_Gabungan_Sebelum_Dedup.csv` adalah:

> Gabungan seluruh data berstatus `KEEP` dari Dwi, Indra, dan Rajif setelah penerapan keputusan review entitas, tetapi sebelum deduplikasi lintas anggota.

Kolom `anggota` menunjukkan **sumber pengumpulan data**, bukan pemilik usaha.

Duplikasi lintas anggota masih mungkin ditemukan karena listing yang sama dapat diperoleh oleh lebih dari satu anggota atau keyword. Deduplikasi tersebut dilakukan pada notebook berikutnya.


In [ ]:

# ============================================================
# 12. MENYIAPKAN FILE OUTPUT PER ANGGOTA
# ============================================================
output_files = []

for anggota in member_order:
    df_keep_audit = kept_by_member[anggota].copy()
    df_drop_audit = dropped_by_member[anggota].copy()

    audit_output = f"02_Data_Reviewed_{anggota}_Audit.csv"
    model_output = f"02_Data_Reviewed_{anggota}_Model.csv"
    drop_output = f"02_Drop_Audit_Entitas_{anggota}.csv"

    df_keep_audit.to_csv(
        audit_output,
        index=False,
        sep=";",
        encoding="utf-8-sig",
        decimal=","
    )

    df_keep_audit[MODEL_COLUMNS].to_csv(
        model_output,
        index=False,
        sep=";",
        encoding="utf-8-sig",
        decimal=","
    )

    df_drop_audit.to_csv(
        drop_output,
        index=False,
        sep=";",
        encoding="utf-8-sig",
        decimal=","
    )

    output_files.extend([
        audit_output,
        model_output,
        drop_output,
    ])

df_gabungan.to_csv(
    OUTPUT_COMBINED_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig",
    decimal=","
)

ringkasan_review.to_csv(
    OUTPUT_SUMMARY_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

output_files.extend([
    OUTPUT_COMBINED_CSV,
    OUTPUT_SUMMARY_CSV,
])

print("CSV hasil review dan penggabungan berhasil dibuat.")


In [ ]:

# ============================================================
# 13. MEMBUAT WORKBOOK BUKTI
# ============================================================
# Export ke CSV
kriteria_operasional.to_csv(f"output_Kriteria_Operasional.csv",
        index=False
    )

    master.to_csv(f"output_Master_Keputusan.csv",
        index=False
    )

    coverage_summary.to_csv(f"output_Cakupan_Master.csv",
        index=False
    )

    unused_master.to_csv(f"output_Master_Tidak_Dipakai.csv",
        index=False
    )

    ringkasan_review_lengkap.to_csv(f"output_Ringkasan_Review.csv",
        index=False
    )

    checkpoint_validation.to_csv(f"output_Validasi_Checkpoint.csv",
        index=False
    )

    for anggota in member_order:
        kept_by_member[anggota].to_csv(
            sheet_name=f"KEEP_{anggota}",
            index=False
        )

        dropped_by_member[anggota].to_csv(
            sheet_name=f"DROP_{anggota}",
            index=False
        )

    df_gabungan.to_csv(f"output_Gabungan_Sebelum_Dedup.csv",
        index=False
    )

output_files.append(OUTPUT_EVIDENCE_CSV)

print("Workbook bukti berhasil dibuat:", OUTPUT_EVIDENCE_CSV)


In [ ]:

# ============================================================
# 14. VALIDASI AKHIR
# ============================================================
assert ringkasan_review_lengkap["siap_merger_awal"].sum() == 3575
assert ringkasan_review_lengkap["drop_final"].sum() == 111
assert ringkasan_review_lengkap["keep_final"].sum() == 3464
assert len(df_gabungan) == 3464
assert df_gabungan["final_decision"].eq("KEEP").all()
assert df_gabungan["title"].notna().all()
assert df_gabungan["text"].notna().all()

for anggota in member_order:
    assert len(kept_by_member[anggota]) == EXPECTED_COUNTS[anggota]["keep_final"]
    assert len(dropped_by_member[anggota]) == EXPECTED_COUNTS[anggota]["review_drop"]

print("SEMUA VALIDASI AKHIR LULUS.")
print("Jumlah final sebelum deduplikasi lintas anggota: 3.464")


In [ ]:

# ============================================================
# 15. MEMBUAT PAKET ZIP
# ============================================================
# Master keputusan disertakan sebagai bukti input keputusan manual.
files_for_zip = [
    MASTER_DECISION_FILE,
    *output_files,
]

with zipfile.ZipFile(
    OUTPUT_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED
) as zip_file:

    for file_name in files_for_zip:
        if os.path.exists(file_name):
            zip_file.write(
                file_name,
                arcname=os.path.basename(file_name)
            )

print("Paket bukti berhasil dibuat:", OUTPUT_ZIP)


In [ ]:
if RUNNING_IN_COLAB:
    files.download(OUTPUT_COMBINED_CSV)
    files.download(OUTPUT_SUMMARY_CSV)
    files.download(OUTPUT_EVIDENCE_CSV)
else:
    print("File tersimpan pada folder kerja.")



## Tahap berikutnya

Gunakan:

`02_Data_Reviewed_Gabungan_Sebelum_Dedup.csv`

sebagai input untuk:

`02_Merger_dan_Audit_Lintas_Anggota_Final.ipynb`

Jangan menjalankan NLP sebelum deduplikasi lintas anggota selesai, karena satu listing masih dapat muncul dari lebih dari satu sumber pengumpulan.
